In [2]:
import yaml
import pickle
import numpy as np
from model import MASTER
from trainer import Trainer
import torch
from torch.utils.data import DataLoader
import pandas as pd

class DailyBatchSampler:
    def __init__(self, data_source, shuffle=False):
        self.data_source = data_source
        self.shuffle = shuffle
        self.daily_count = pd.Series(index=data_source.get_index()).groupby("datetime").size().values
        self.daily_index = np.roll(np.cumsum(self.daily_count), 1)
        self.daily_index[0] = 0

    def __iter__(self):
        indices = np.arange(len(self.daily_count))
        if self.shuffle: np.random.shuffle(indices)
        for i in indices:
            yield np.arange(self.daily_index[i], self.daily_index[i] + self.daily_count[i])

    def __len__(self):
        return len(self.data_source)

def load_config(file='config.yaml'):
    with open(file, 'r') as f:
        return yaml.safe_load(f)
 
def load_data(config):
    data_cfg = config['data']
    with open(f"{data_cfg['train_data_dir']}/{data_cfg['prefix']}/{data_cfg['universe']}_dl_train.pkl", 'rb') as f:
        dl_train = pickle.load(f)
    with open(f"{data_cfg['predict_data_dir']}/{data_cfg['universe']}_dl_valid.pkl", 'rb') as f:
        dl_valid = pickle.load(f)
    with open(f"{data_cfg['predict_data_dir']}/{data_cfg['universe']}_dl_test.pkl", 'rb') as f:
        dl_test = pickle.load(f)
    return dl_train, dl_valid, dl_test

config = load_config()

if config['data']['universe'] == 'csi300':
    config['model']['beta'] = 5
elif config['data']['universe'] == 'csi800':
    config['model']['beta'] = 2

dl_train, dl_valid, dl_test = load_data(config)

ic, icir, ric, ricir = [], [], [], []
loder = DataLoader(dl_test, sampler=DailyBatchSampler(dl_test, False), drop_last=False)


In [ ]:
import numpy as np
np.isnan()